In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
model_path = Path("/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/calibration/gmr_dayblock_final.pkl")

with open(model_path, "rb") as f:
    gmm_loaded = pickle.load(f)

print(f"Loaded model from: {model_path}")

Loaded model from: /Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/calibration/gmr_dayblock_final.pkl


In [4]:
# ---- PM2.5 ----
pm25_files = [
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/pm/summarized/pm25_community_hourly_20231024-20240816.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/pm/summarized/pm25_community_hourly_20240816-20241031.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/pm/summarized/pm25_community_hourly_20241031-20250901.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/pm/summarized/pm25_community_hourly_20250831-20260315.csv"
]

pm25 = pd.concat([pd.read_csv(f) for f in pm25_files], ignore_index=True)


# ---- Temperature ----
temp_files = [
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/weather/summarized/temp_hourly_20230815-20240820.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/weather/summarized/temp_hourly_20240816-20241031.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/weather/summarized/temp_hourly_20241031-20250901.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/weather/summarized/temp_hourly_20250831-20260315.csv"
]

temp = pd.concat([pd.read_csv(f) for f in temp_files], ignore_index=True)


# ---- RH ----
rh_files = [
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/weather/summarized/rh_hourly_20230815-20240820.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/weather/summarized/rh_hourly_20240816-20241031.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/weather/summarized/rh_hourly_20241031-20250901.csv",
    "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/weather/summarized/rh_hourly_20250831-20260315.csv"
]

rh = pd.concat([pd.read_csv(f) for f in rh_files], ignore_index=True)

pm25 = pm25.drop_duplicates()
temp = temp.drop_duplicates()
rh   = rh.drop_duplicates()

In [5]:
# keep non-missing measurements
pm25 = pm25.loc[pm25["mean_pm25"].notna()].copy()
temp = temp.loc[temp["mean_met_temp"].notna()].copy()
rh   = rh.loc[rh["mean_met_rh"].notna()].copy()

# drop extra columns if present
for df, cols in [
    (pm25, ["n_minute_obs", "n_active", "fleet_average_pm25"]),
    (temp, ["n_minute_obs", "n_active", "fleet_average_met_temp"]),
    (rh,   ["n_minute_obs", "n_active", "fleet_average_met_rh"]),
]:
    drop_cols = [c for c in cols if c in df.columns]
    df.drop(columns=drop_cols, inplace=True)

# merge
modulair_apply = (
    pm25.merge(temp, on=["monitor", "date", "hour"], how="outer")
        .merge(rh, on=["monitor", "date", "hour"], how="outer")
)

# rename columns to match model inputs
modulair_apply = modulair_apply.rename(columns={
    "mean_pm25": "mod_pm25",
    "mean_met_temp": "mod_temp",
    "mean_met_rh": "mod_rh"
})

In [6]:
# parse date
modulair_apply["date"] = pd.to_datetime(modulair_apply["date"])

# make datetime if useful
modulair_apply["date_hour"] = pd.to_datetime(
    modulair_apply["date"].dt.strftime("%Y-%m-%d") + " " +
    modulair_apply["hour"].astype(int).astype(str) + ":00:00",
    utc=True
)

# cyclic hour variables
modulair_apply["sin_hour"] = np.sin(2 * np.pi * modulair_apply["hour"] / 24)
modulair_apply["cos_hour"] = np.cos(2 * np.pi * modulair_apply["hour"] / 24)

# season variable: Harmattan = Dec-Feb
modulair_apply["month"] = modulair_apply["date"].dt.month
modulair_apply["season_binary"] = np.where(modulair_apply["month"].isin([12, 1, 2]), 1, 0)

In [7]:
x_cols = ["mod_pm25", "mod_temp", "mod_rh", "sin_hour", "cos_hour", "season_binary"]

modulair_apply_valid = modulair_apply.dropna(subset=x_cols).copy()

print("Rows available for calibration:", len(modulair_apply_valid))

Rows available for calibration: 556642


In [8]:
X_full = modulair_apply_valid[x_cols].to_numpy()

Y_full = gmm_loaded.predict(
    np.array([0, 1, 2, 3, 4, 5]),
    X_full
).ravel()

modulair_apply_valid["pm25_fem_gmr"] = np.maximum(Y_full, 0)
modulair_apply_valid["delta_gmr"] = modulair_apply_valid["pm25_fem_gmr"] - modulair_apply_valid["mod_pm25"]
modulair_apply_valid["ratio_gmr"] = np.where(
    modulair_apply_valid["mod_pm25"] > 0,
    modulair_apply_valid["pm25_fem_gmr"] / modulair_apply_valid["mod_pm25"],
    np.nan
)

modulair_apply_valid.head()

,monitor,date,hour,mod_pm25,mod_temp,mod_rh,date_hour,sin_hour,cos_hour,month,season_binary,pm25_fem_gmr,delta_gmr,ratio_gmr
0,MOD-00077,2024-09-03,16.0,65.614000,35.835000,46.395000,2024-09-03 16:00:00+00:00,-0.866025,-5.000000e-01,9,0,24.579372,-41.034628,0.374606
1,MOD-00077,2024-09-03,17.0,102.075000,32.908333,53.998333,2024-09-03 17:00:00+00:00,-0.965926,-2.588190e-01,9,0,35.716478,-66.358522,0.349904
2,MOD-00077,2024-09-03,18.0,153.113500,30.603333,62.325000,2024-09-03 18:00:00+00:00,-1.000000,-1.836970e-16,9,0,51.661698,-101.451802,0.337408
3,MOD-00077,2024-09-03,19.0,151.813000,29.363333,68.226667,2024-09-03 19:00:00+00:00,-0.965926,2.588190e-01,9,0,52.138254,-99.674746,0.343437
4,MOD-00077,2024-09-03,20.0,82.144833,28.330000,72.103333,2024-09-03 20:00:00+00:00,-0.866025,5.000000e-01,9,0,32.040908,-50.103925,0.390054


In [9]:
pred_cols = ["pm25_fem_gmr", "delta_gmr", "ratio_gmr"]

modulair_apply.loc[modulair_apply_valid.index, pred_cols] = modulair_apply_valid[pred_cols]

modulair_apply

,monitor,date,hour,mod_pm25,mod_temp,mod_rh,date_hour,sin_hour,cos_hour,month,season_binary,pm25_fem_gmr,delta_gmr,ratio_gmr
0,MOD-00077,2024-09-03,16.0,65.614000,35.835000,46.395000,2024-09-03 16:00:00+00:00,-0.866025,-5.000000e-01,9,0,24.579372,-41.034628,0.374606
1,MOD-00077,2024-09-03,17.0,102.075000,32.908333,53.998333,2024-09-03 17:00:00+00:00,-0.965926,-2.588190e-01,9,0,35.716478,-66.358522,0.349904
2,MOD-00077,2024-09-03,18.0,153.113500,30.603333,62.325000,2024-09-03 18:00:00+00:00,-1.000000,-1.836970e-16,9,0,51.661698,-101.451802,0.337408
3,MOD-00077,2024-09-03,19.0,151.813000,29.363333,68.226667,2024-09-03 19:00:00+00:00,-0.965926,2.588190e-01,9,0,52.138254,-99.674746,0.343437
4,MOD-00077,2024-09-03,20.0,82.144833,28.330000,72.103333,2024-09-03 20:00:00+00:00,-0.866025,5.000000e-01,9,0,32.040908,-50.103925,0.390054
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699510,MOD-PM-01093,2026-03-12,5.0,7.015866,27.834821,71.684643,2026-03-12 05:00:00+00:00,0.965926,2.588190e-01,3,0,12.028235,5.012369,1.714433
699511,MOD-PM-01093,2026-03-13,7.0,5.870465,30.366604,65.292075,2026-03-13 07:00:00+00:00,0.965926,-2.588190e-01,3,0,10.510674,4.640210,1.790433
699512,MOD-PM-01093,2026-03-13,8.0,8.969727,32.395357,59.195893,2026-03-13 08:00:00+00:00,0.866025,-5.000000e-01,3,0,10.731467,1.761740,1.196410
699513,MOD-PM-01093,2026-03-13,9.0,8.140360,34.749322,52.739661,2026-03-13 09:00:00+00:00,0.707107,-7.071068e-01,3,0,9.829280,1.688920,1.207475


In [ ]:
outpath = "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/pm/GRIMM_calibrated_pm_summaries/hourly_modulair_full_calibrated_gmr_20231024-20260315.csv"
modulair_apply.to_csv(outpath, index=False)

print(f"Saved calibrated data to: {outpath}")

Saved calibrated data to: /Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/pm/GRIMM_calibrated_pm_summaries/modulair_full_calibrated_gmr_20231024-20260315.csv


In [11]:

# make sure date is datetime
modulair_apply["date"] = pd.to_datetime(modulair_apply["date"])

daily_summary = (
    modulair_apply
    .groupby(["monitor", "date"], as_index=False)
    .agg(
        n_complete_hours_raw=("mod_pm25", lambda x: x.notna().sum()),
        mean_mod_pm25=("mod_pm25", lambda x: x.mean(skipna=True)),
        
        n_complete_hours_gmr=("pm25_fem_gmr", lambda x: x.notna().sum()),
        mean_pm25_fem_gmr=("pm25_fem_gmr", lambda x: x.mean(skipna=True))
    )
)

# apply >=18 hour rule separately to raw and calibrated summaries
daily_summary["mean_mod_pm25"] = np.where(
    daily_summary["n_complete_hours_raw"] >= 18,
    daily_summary["mean_mod_pm25"],
    np.nan
)

daily_summary["mean_pm25_fem_gmr"] = np.where(
    daily_summary["n_complete_hours_gmr"] >= 18,
    daily_summary["mean_pm25_fem_gmr"],
    np.nan
)

# optional comparison columns
daily_summary["delta_daily_gmr"] = (
    daily_summary["mean_pm25_fem_gmr"] - daily_summary["mean_mod_pm25"]
)

daily_summary["ratio_daily_gmr"] = np.where(
    daily_summary["mean_mod_pm25"] > 0,
    daily_summary["mean_pm25_fem_gmr"] / daily_summary["mean_mod_pm25"],
    np.nan
)

daily_summary.head()

,monitor,date,n_complete_hours_raw,mean_mod_pm25,n_complete_hours_gmr,mean_pm25_fem_gmr,delta_daily_gmr,ratio_daily_gmr
0,MOD-00077,2024-09-03,8,NaN,8,NaN,NaN,NaN
1,MOD-00077,2024-09-04,24,44.504806,24,21.910068,-22.594737,0.492308
2,MOD-00077,2024-09-05,24,45.187292,24,21.409635,-23.777657,0.473798
3,MOD-00077,2024-09-06,21,46.421955,21,22.805066,-23.616889,0.491256
4,MOD-00077,2024-09-07,24,59.199493,24,25.377346,-33.822147,0.428675


In [12]:
daily_outpath = "/Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/pm/GRIMM_calibrated_pm_summaries/daily_modulair_full_calibrated_gmr_20231024-20260315.csv"
daily_summary.to_csv(daily_outpath, index=False)

print(f"Saved daily summary to: {daily_outpath}")

Saved daily summary to: /Users/lewiswhite/CHAP_columbia/QuantAQ_ghana/data/pm/GRIMM_calibrated_pm_summaries/daily_modulair_full_calibrated_gmr_20231024-20260315.csv
